# All comparisons of TDC tasks

In [ ]:
from pathlib import Path
import json
from collections import defaultdict
import pandas as pd
import numpy as np

In [ ]:
# Define paths
OUR_RESULTS_DIR = Path("../../data/results/tdc_tasks")
PMO_10K_RESULTS_DIR = Path("../../data/results/pmo_baseline")
PMO_50_RESULTS_CSV = Path("../../data/results/tdc_top_50_baselines/results.csv")

# Define model mappings from filename to display name (PMO baselines)
MODEL_MAPPING_10K = {
    'reinvent': 'REINVENT',
    'reinvent_selfies': 'REINVENT SELFIES',
    'graph_ga': 'Graph GA',
    'gp_bo': 'GP BO',
    'stoned': 'STONED',
}

MODEL_MAPPING_50 = {
    'reinvent': 'REINVENT',
    'reinvent_selfies': 'REINVENT SELFIES',
    'graph_ga': 'Graph GA',
    'gpbo': 'GP BO',
}

# Our model name
OUR_MODEL_NAME = 'LLM Agent (Ours)'

# List of all 23 PMO benchmark tasks
TASKS = [
    'albuterol_similarity',
    'amlodipine_mpo',
    'celecoxib_rediscovery',
    'deco_hop',
    'drd2',
    'fexofenadine_mpo',
    'gsk3b',
    'isomers_c7h8n2o2',
    'isomers_c9h10n2o2pf2cl',
    'jnk3',
    'median1',
    'median2',
    'mestranol_similarity',
    'osimertinib_mpo',
    'perindopril_mpo',
    'qed',
    'ranolazine_mpo',
    'scaffold_hop',
    'sitagliptin_mpo',
    'thiothixene_rediscovery',
    'troglitazone_rediscovery',
    'valsartan_smarts',
    'zaleplon_mpo',
]

## Loading our data (LLM Agent)

In [ ]:
def load_trace(p: Path) -> list:
    """Load trace from JSON file."""
    d = json.loads(p.read_text(encoding="utf-8"))
    return d["trace"] if isinstance(d, dict) and "trace" in d else d


def load_our_results_as_df(results_dir: Path) -> pd.DataFrame:
    """Load all our results from JSON trace files into a DataFrame.
    
    Returns DataFrame with columns: model, task, run, score, oracle_call
    """
    records = []
    
    for task_dir in results_dir.iterdir():
        if not task_dir.is_dir():
            continue
        
        task_name = task_dir.name
        
        for run_idx, json_file in enumerate(sorted(task_dir.glob("*.json"))):
            try:
                trace = load_trace(json_file)
                for entry in trace:
                    records.append({
                        'model': OUR_MODEL_NAME,
                        'task': task_name,
                        'run': run_idx,
                        'score': float(entry["score"]),
                        'oracle_call': int(entry["iteration"])
                    })
            except Exception as e:
                print(f"Error loading {json_file}: {e}")
    
    return pd.DataFrame(records)


# Load our results
our_df = load_our_results_as_df(OUR_RESULTS_DIR)
print(f"Our results: {len(our_df)} rows, {our_df['task'].nunique()} tasks, {our_df.groupby('task')['run'].nunique().max()} max runs")
our_df.head()

## Loading PMO 10K results (for AUC over 10000 iterations)

In [ ]:
def load_yaml_results_fast(filepath):
    """Load results from a YAML file using fast custom parsing."""
    results = []
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    i = 0
    n = len(lines)
    while i < n:
        line = lines[i]
        if not line.strip():
            i += 1
            continue
        if not line.startswith('-'):
            if i + 2 < n:
                score_line = lines[i + 1].strip()
                oracle_line = lines[i + 2].strip()
                if score_line.startswith('- ') and oracle_line.startswith('- '):
                    try:
                        score = float(score_line[2:])
                        oracle_call = int(oracle_line[2:])
                        results.append((score, oracle_call))
                    except (ValueError, IndexError):
                        pass
                    i += 3
                    continue
        i += 1
    
    results.sort(key=lambda x: x[1])
    return results


def load_pmo_10k_results_as_df(results_dir: Path, model_mapping: dict) -> pd.DataFrame:
    """Load raw PMO results from YAML files into a DataFrame."""
    records = []
    
    for model_key, model_name in model_mapping.items():
        for task in TASKS:
            pattern = f"results_{model_key}_{task}_*.yaml"
            for run_idx, filepath in enumerate(sorted(results_dir.glob(pattern))):
                results = load_yaml_results_fast(filepath)
                for score, oracle_call in results:
                    records.append({
                        'model': model_name,
                        'task': task,
                        'run': run_idx,
                        'score': score,
                        'oracle_call': oracle_call
                    })
    
    return pd.DataFrame(records)


print("Loading PMO baseline results (10K oracle calls) from YAML files...")
pmo_10k_df = load_pmo_10k_results_as_df(PMO_10K_RESULTS_DIR, MODEL_MAPPING_10K)
print(f"PMO 10K results: {len(pmo_10k_df)} rows, {pmo_10k_df['model'].nunique()} models")
print(f"Runs per model: {pmo_10k_df.groupby('model')['run'].nunique().to_dict()}")
pmo_10k_df.head()

## Loading PMO 50-iteration results (for AUC over 50 iterations)

In [ ]:
def load_pmo_50_results_as_df(csv_path: Path, model_mapping: dict) -> pd.DataFrame:
    """Load PMO 50-iteration results from CSV into a DataFrame.
    
    CSV columns: model, task, repeat, iteration, molecule, score, rank, duration_seconds, timestamp
    """
    df = pd.read_csv(csv_path)
    
    # Rename columns to match our format
    df = df.rename(columns={
        'repeat': 'run',
        'iteration': 'oracle_call'
    })
    
    # Map model names
    df['model'] = df['model'].map(model_mapping)
    
    # Keep only relevant columns
    df = df[['model', 'task', 'run', 'score', 'oracle_call']].copy()
    
    # Drop rows with unmapped models
    df = df.dropna(subset=['model'])
    
    return df


print("Loading PMO baseline results (50 iterations) from CSV...")
pmo_50_df = load_pmo_50_results_as_df(PMO_50_RESULTS_CSV, MODEL_MAPPING_50)
print(f"PMO 50 results: {len(pmo_50_df)} rows, {pmo_50_df['model'].nunique()} models")
print(f"Tasks covered: {sorted(pmo_50_df['task'].unique())}")
pmo_50_df.head()

---
## AUC Top-1 over 10000 iterations

In [ ]:
def compute_auc_top1_from_df(group_df, max_oracle_calls=10000):
    """Compute AUC of top-1 (best) score vs oracle calls from a dataframe group.
    
    Args:
        group_df: DataFrame with 'score' and 'oracle_call' columns for a single run
        max_oracle_calls: Maximum number of oracle calls (default 10000)
    
    Returns:
        AUC value normalized to [0, 1]
    """
    sorted_df = group_df.sort_values('oracle_call')
    
    best_score = 0.0
    best_at_call = {}
    
    for _, row in sorted_df.iterrows():
        oracle_call = int(row['oracle_call'])
        score = float(row['score'])
        
        if oracle_call > max_oracle_calls:
            break
        
        if score > best_score:
            best_score = score
        
        best_at_call[oracle_call] = best_score
    
    oracle_calls = sorted(best_at_call.keys())
    
    # Compute AUC as sum of rectangles
    auc = 0.0
    prev_call = 0
    prev_best = 0.0
    
    for call in oracle_calls:
        auc += prev_best * (call - prev_call)
        prev_call = call
        prev_best = best_at_call[call]
    
    # Extend to max_oracle_calls
    auc += prev_best * (max_oracle_calls - prev_call)
    
    return auc / max_oracle_calls


def create_comparison_table(auc_summary, model_sums, tasks, model_order):
    """Create a comparison table DataFrame from AUC summary."""
    rows = []
    for task in tasks:
        row = {'Task': task}
        task_data = auc_summary[auc_summary['task'] == task]
        
        for model in model_order:
            model_data = task_data[task_data['model'] == model]
            if len(model_data) > 0:
                mean_val = model_data['auc_mean'].values[0]
                std_val = model_data['auc_std'].values[0]
                if pd.isna(std_val) or std_val == 0:
                    row[model] = f"{mean_val:.3f}"
                else:
                    row[model] = f"{mean_val:.3f}± {std_val:.3f}"
            else:
                row[model] = "-"
        rows.append(row)
    
    # Add Sum row
    sum_row = {'Task': 'Sum'}
    for model in model_order:
        model_sum = model_sums[model_sums['model'] == model]['sum_auc'].values[0]
        sum_row[model] = f"{model_sum:.3f}"
    rows.append(sum_row)
    
    # Add Rank row
    rank_row = {'Task': 'Rank'}
    for i, model in enumerate(model_order):
        rank_row[model] = str(i + 1)
    rows.append(rank_row)
    
    table = pd.DataFrame(rows)
    return table[['Task'] + model_order]

In [ ]:
# Combine our results with PMO 10K results
df_10k = pd.concat([our_df, pmo_10k_df], ignore_index=True)
print(f"Combined 10K dataframe: {len(df_10k)} rows, {df_10k['model'].nunique()} models")

# Compute AUC Top-1 for each model/task/run
MAX_ORACLE_CALLS = 10000

auc_records = []
for (model, task, run), group in df_10k.groupby(['model', 'task', 'run']):
    auc = compute_auc_top1_from_df(group, MAX_ORACLE_CALLS)
    auc_records.append({
        'model': model,
        'task': task,
        'run': run,
        'auc_top1': auc
    })

auc_df = pd.DataFrame(auc_records)
print(f"Computed AUC Top-1 for {len(auc_df)} model/task/run combinations")

# Compute mean and std AUC per model/task
auc_summary = auc_df.groupby(['model', 'task'])['auc_top1'].agg(['mean', 'std']).reset_index()
auc_summary.columns = ['model', 'task', 'auc_mean', 'auc_std']

# Compute sum of mean AUC across all tasks for each model
model_sums = auc_summary.groupby('model')['auc_mean'].sum().reset_index()
model_sums.columns = ['model', 'sum_auc']
model_sums = model_sums.sort_values('sum_auc', ascending=False).reset_index(drop=True)
model_sums['rank'] = range(1, len(model_sums) + 1)

print(f"\nTop models by sum of AUC Top-1 (10K iterations):")
print(model_sums.to_string(index=False))

In [ ]:
# Create comparison table for 10K iterations
model_order = model_sums['model'].tolist()
comparison_table_10k = create_comparison_table(auc_summary, model_sums, TASKS, model_order)

print("AUC Top-1 over 10000 iterations:")
comparison_table_10k

---
## AUC Top-1 over 50 iterations

In [ ]:
# Combine our results with PMO 50 results
df_50 = pd.concat([our_df, pmo_50_df], ignore_index=True)
print(f"Combined 50-iter dataframe: {len(df_50)} rows, {df_50['model'].nunique()} models")

# Compute AUC Top-1 for each model/task/run with 50 iterations
MAX_ORACLE_CALLS_50 = 50

auc_records_50 = []
for (model, task, run), group in df_50.groupby(['model', 'task', 'run']):
    auc = compute_auc_top1_from_df(group, MAX_ORACLE_CALLS_50)
    auc_records_50.append({
        'model': model,
        'task': task,
        'run': run,
        'auc_top1': auc
    })

auc_df_50 = pd.DataFrame(auc_records_50)
print(f"Computed AUC Top-1 (50 iterations) for {len(auc_df_50)} model/task/run combinations")

# Compute mean and std AUC per model/task
auc_summary_50 = auc_df_50.groupby(['model', 'task'])['auc_top1'].agg(['mean', 'std']).reset_index()
auc_summary_50.columns = ['model', 'task', 'auc_mean', 'auc_std']

# Compute sum of mean AUC across all tasks for each model
model_sums_50 = auc_summary_50.groupby('model')['auc_mean'].sum().reset_index()
model_sums_50.columns = ['model', 'sum_auc']
model_sums_50 = model_sums_50.sort_values('sum_auc', ascending=False).reset_index(drop=True)
model_sums_50['rank'] = range(1, len(model_sums_50) + 1)

print(f"\nTop models by sum of AUC Top-1 (50 iterations):")
print(model_sums_50.to_string(index=False))

In [ ]:
# Create comparison table for 50 iterations
model_order_50 = model_sums_50['model'].tolist()
comparison_table_50 = create_comparison_table(auc_summary_50, model_sums_50, TASKS, model_order_50)

print("AUC Top-1 over 50 iterations:")
comparison_table_50

---
## Best Score after 50 iterations

In [ ]:
def compute_best_score(group_df, max_oracle_calls=50):
    """Compute the best score achieved within max_oracle_calls."""
    filtered = group_df[group_df['oracle_call'] <= max_oracle_calls]
    if len(filtered) == 0:
        return 0.0
    return filtered['score'].max()


# Compute best score for each model/task/run
MAX_ITERATIONS_BEST = 50

best_score_records = []
for (model, task, run), group in df_50.groupby(['model', 'task', 'run']):
    best = compute_best_score(group, MAX_ITERATIONS_BEST)
    best_score_records.append({
        'model': model,
        'task': task,
        'run': run,
        'best_score': best
    })

best_score_df = pd.DataFrame(best_score_records)
print(f"Computed best score (50 iterations) for {len(best_score_df)} model/task/run combinations")

# Compute mean and std best score per model/task
best_score_summary = best_score_df.groupby(['model', 'task'])['best_score'].agg(['mean', 'std']).reset_index()
best_score_summary.columns = ['model', 'task', 'score_mean', 'score_std']

# Compute sum of mean best score across all tasks for each model
model_sums_best = best_score_summary.groupby('model')['score_mean'].sum().reset_index()
model_sums_best.columns = ['model', 'sum_score']
model_sums_best = model_sums_best.sort_values('sum_score', ascending=False).reset_index(drop=True)
model_sums_best['rank'] = range(1, len(model_sums_best) + 1)

print(f"\nTop models by sum of Best Score (50 iterations):")
print(model_sums_best.to_string(index=False))

In [ ]:
# Create comparison table for best score
def create_best_score_table(score_summary, model_sums, tasks, model_order):
    """Create a comparison table DataFrame from best score summary."""
    rows = []
    for task in tasks:
        row = {'Task': task}
        task_data = score_summary[score_summary['task'] == task]
        
        for model in model_order:
            model_data = task_data[task_data['model'] == model]
            if len(model_data) > 0:
                mean_val = model_data['score_mean'].values[0]
                std_val = model_data['score_std'].values[0]
                if pd.isna(std_val) or std_val == 0:
                    row[model] = f"{mean_val:.3f}"
                else:
                    row[model] = f"{mean_val:.3f}± {std_val:.3f}"
            else:
                row[model] = "-"
        rows.append(row)
    
    # Add Sum row
    sum_row = {'Task': 'Sum'}
    for model in model_order:
        model_sum = model_sums[model_sums['model'] == model]['sum_score'].values[0]
        sum_row[model] = f"{model_sum:.3f}"
    rows.append(sum_row)
    
    # Add Rank row
    rank_row = {'Task': 'Rank'}
    for i, model in enumerate(model_order):
        rank_row[model] = str(i + 1)
    rows.append(rank_row)
    
    table = pd.DataFrame(rows)
    return table[['Task'] + model_order]


model_order_best = model_sums_best['model'].tolist()
comparison_table_best = create_best_score_table(best_score_summary, model_sums_best, TASKS, model_order_best)

print("Best Score after 50 iterations:")
comparison_table_best